# GOTSF Fig. 5 데이터 진단 · v7

저장소의 `dataset/wireless/wireless.csv`(BLW-TrafficDS, 1,026시간 × 100 beam)가 논문이 공개한 Hugging Face 데이터셋(`netop/Beam-Level-Traffic-Timeseries-Dataset`)의 어느 부분에서 왔는지 확인합니다. v6에서 DLPRB 6주치와 비교했을 때는 정확히 일치하는 beam이 없었습니다(상관계수 중앙값 0.58).

**확인하는 것**
1. **지표**: DLPRB(PRB 활용률), DLThpVol(처리량), DLThpTime(처리 시간), MR_number(측정 보고 수) 4가지 모두
2. **기간**: 0–5주(학습), 5–6주, 10–11주 파일과, 0–6주를 이어 붙인 것
3. **합산 단위**: 개별 beam뿐 아니라 beam 이름(`a_b_c`)의 앞부분이 같은 것끼리 합친 계열(`a_b` 단위, `a` 단위)

`wireless.csv`의 beam 10개를 골라, 시간을 한 시간씩 밀어 가며 모든 계열과의 상관계수를 계산합니다. 정확히 같은 계열이면 상관계수가 1이 됩니다(크기만 다른 것은 상관계수에 영향이 없음).

**실행**: GPU가 필요 없어 CPU 런타임으로 충분하고, 데이터 내려받기를 포함해 5–10분 걸립니다. **마지막 셀의 출력(표와 결론)을 복사해서 보내주세요.**

In [1]:
import os, time, numpy as np, pandas as pd
HF_REPO = "netop/Beam-Level-Traffic-Timeseries-Dataset"
LOCAL_DIR = None    # (점검용) Hugging Face 대신 이 폴더에서 같은 이름의 파일을 읽음
MODALITIES = ["DLPRB", "DLThpVol", "DLThpTime", "MR_number"]
FILES = {"0-5주": "data/train/{m}_train_0w-5w.csv", "5-6주": "data/test/{m}_test_5w-6w.csv", "10-11주": "data/test/{m}_test_10w-11w.csv"}
WIRELESS_URL = ("https://raw.githubusercontent.com/netop-team/gotsf/31b17e55a0cb6f41bfe25230db3f81567efd58f3/"
                "dataset/wireless/wireless.csv")

W = pd.read_csv(WIRELESS_URL).iloc[:, 1:].to_numpy(float)
TEST_COLS = list(range(0, W.shape[1], 10))          # wireless.csv의 beam 10개로 확인
print(f"wireless.csv {W.shape} | 전체 평균 {W.mean():.3f} (전체를 하나의 값으로 나눈 데이터), beam별 평균 {W.mean(0).min():.3f}~{W.mean(0).max():.3f}")

def load(mod, part):
    name = FILES[part].format(m=mod)
    if LOCAL_DIR:
        path = os.path.join(LOCAL_DIR, os.path.basename(name))
    else:
        from huggingface_hub import hf_hub_download
        path = hf_hub_download(HF_REPO, name, repo_type="dataset")
    df = pd.read_csv(path)
    return df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")]).fillna(0.0)

def aggregate(df, level):                             # beam 이름 "a_b_c"의 앞 level개 부분이 같은 것끼리 합산
    keys = ["_".join(c.split("_")[:level]) for c in df.columns]
    return df.T.groupby(keys).sum().T

wireless.csv (1026, 100) | 전체 평균 1.000 (전체를 하나의 값으로 나눈 데이터), beam별 평균 0.012~7.039


In [2]:
def zs(a):
    return (a - a.mean(0)) / (a.std(0) + 1e-12)

def best_match(H, min_overlap=400):
    # 시간 차이(lag = H 시작 - W 시작)마다 W의 beam 10개와 H의 모든 계열 사이 상관계수를 계산해 가장 잘 맞는 lag를 찾음
    T, S = len(W), len(H); m = min(min_overlap, S, T)
    best_c, best_lag = np.full(len(TEST_COLS), -1.0), None
    for lag in range(-(T - m), S - m + 1):
        w0, h0 = max(0, -lag), max(0, lag); L = min(T - w0, S - h0)
        c = np.nanmax(zs(W[w0:w0 + L, TEST_COLS]).T @ zs(H[h0:h0 + L]) / L, axis=1)
        if np.median(c) > np.median(best_c):
            best_c, best_lag = c, lag
    return best_c, best_lag

rows, t0 = [], time.time()
for mod in MODALITIES:
    parts = {p: load(mod, p) for p in FILES}
    seqs = {p: (df, "beam") for p, df in parts.items()}
    joined = pd.concat([parts["0-5주"], parts["5-6주"]], ignore_index=True)
    seqs["0-6주 이어 붙임"] = (joined, "beam")
    seqs["0-6주, a_b 단위 합"] = (aggregate(joined, 2), "a_b 합")
    seqs["0-6주, a 단위 합"] = (aggregate(joined, 1), "a 합")
    for name, (df, unit) in seqs.items():
        c, lag = best_match(df.to_numpy(float))
        rows.append({"지표": mod, "기간/단위": name, "계열 수": df.shape[1], "길이(시간)": len(df),
                     "상관계수 중앙값": float(np.median(c)), "상관계수 최솟값": float(c.min()), "시간 차이": lag})
    print(f"✔ {mod} ({time.time() - t0:.0f}s)")
res = pd.DataFrame(rows).sort_values("상관계수 중앙값", ascending=False).reset_index(drop=True)

data/train/DLPRB_train_0w-5w.csv: reconstructing file:   0%|          |  0.00B / 38.1MB            

data/train/DLPRB_train_0w-5w.csv: downloading bytes:           |  0.00B            

DLPRB_test_5w-6w.csv:   0%|          | 0.00/7.66M [00:00<?, ?B/s]

DLPRB_test_10w-11w.csv:   0%|          | 0.00/7.67M [00:00<?, ?B/s]

✔ DLPRB (122s)


data/train/DLThpVol_train_0w-5w.csv: reconstructing file:   0%|          |  0.00B / 35.5MB            

... (10 outputs left out)

data/train/MR_number_train_0w-5w.csv: downloading bytes:           |  0.00B            

MR_number_test_5w-6w.csv:   0%|          | 0.00/7.91M [00:00<?, ?B/s]

MR_number_test_10w-11w.csv:   0%|          | 0.00/7.91M [00:00<?, ?B/s]

✔ MR_number (437s)


In [3]:
# 분포 비교: beam마다 평균으로 나눈 뒤(wireless.csv와 같은 정규화)의 모양
def fingerprint(A):
    m = A.mean(0); A = A[:, m > 0] / m[m > 0]
    return {"0의 비율": (A == 0).mean(), "중앙값": np.median(A), "99% 분위": np.quantile(A, 0.99), "beam별 최댓값(중앙값)": np.median(A.max(0))}
fp = {"wireless.csv": fingerprint(W)}
for mod in MODALITIES:
    fp[f"{mod} (0-6주, beam)"] = fingerprint(pd.concat([load(mod, "0-5주"), load(mod, "5-6주")], ignore_index=True).to_numpy(float))
fp = pd.DataFrame(fp).T

pd.set_option("display.width", 200)
print("== 시계열 일치 검사 (상관계수 1 = 같은 계열) ==")
print(res.round(4).to_string(index=False))
print("\n== 분포 비교 (beam별 평균으로 나눈 뒤) ==")
print(fp.round(4).to_string())
top = res.iloc[0]
print("\n== 결론 ==")
if top["상관계수 중앙값"] >= 0.99:
    print(f"✅ wireless.csv는 {top['지표']}의 '{top['기간/단위']}'와 일치합니다 (시간 차이 {top['시간 차이']}시간, 상관계수 최솟값 {top['상관계수 최솟값']:.4f})")
elif top["상관계수 중앙값"] >= 0.9:
    print(f"◐ {top['지표']}의 '{top['기간/단위']}'와 가깝지만 정확히 일치하지 않습니다 (중앙값 {top['상관계수 중앙값']:.3f}). 평활화 등 가공을 거친 데이터일 수 있습니다")
else:
    print(f"❌ 공개 데이터에서 wireless.csv와 같은 시계열을 찾지 못했습니다 (가장 높은 중앙값 {top['상관계수 중앙값']:.3f}: {top['지표']}, {top['기간/단위']})")

== 시계열 일치 검사 (상관계수 1 = 같은 계열) ==
       지표          기간/단위  계열 수  길이(시간)  상관계수 중앙값  상관계수 최솟값  시간 차이
DLThpTime         10-11주  2880     168    0.6929    0.4495   -216
MR_number         10-11주  2880     168    0.6515    0.4573   -216
    DLPRB         10-11주  2880     168    0.6479    0.4462   -192
MR_number           5-6주  2880     168    0.6359    0.3531   -240
 DLThpVol         10-11주  2880     168    0.6352    0.4959   -216
    DLPRB           5-6주  2880     168    0.6320    0.5025   -168
DLThpTime           5-6주  2880     168    0.6294    0.5043   -144
 DLThpVol           5-6주  2880     168    0.6127    0.2574   -360
    DLPRB     0-6주 이어 붙임  2880    1008    0.5479    0.2775    600
 DLThpVol     0-6주 이어 붙임  2880    1008    0.5369    0.2560    600
MR_number 0-6주, a_b 단위 합    90    1008    0.5320    0.1593    504
MR_number           0-5주  2880     840    0.5319    0.2496    336
MR_number     0-6주 이어 붙임  2880    1008    0.5314    0.2418    600
    DLPRB 0-6주, a_b 단위 합    90    1008    0